In [37]:
import re 
import pandas as pd 
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
import os 

In [38]:
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [39]:
file_path = "data/가전/"

file_list = os.listdir(file_path)

df = pd.DataFrame()

for file in file_list:
    data = pd.read_json(file_path + file)
    df = pd.concat( [df, data], axis= 0 )
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [40]:
df = df[['RawText', 'GeneralPolarity']]

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RawText          4056 non-null   object 
 1   GeneralPolarity  3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 95.1+ KB


In [42]:
df['RawText'] = df['RawText'].map(normalize)

In [43]:
df = df.loc[df['RawText'].str.len() > 1, ]

In [44]:
df.drop_duplicates('RawText', inplace=True)

In [45]:
df.rename(columns = {
    'GeneralPolarity' : 'label'
}, inplace=True)

In [46]:
na_df = df.loc[df['label'].isna()]

In [47]:
df = df.loc[~df['label'].isna()]

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 86.2+ KB


In [49]:
df['label'] = df['label'].map({
    -1 : 0, 
    0 : 1, 
    1 : 2
})

In [50]:
df['label'].value_counts()

label
2    2220
1     944
0     514
Name: count, dtype: int64

In [51]:
train_df, test_df = train_test_split(
    df, test_size = 0.2, random_state = 42, stratify=df['label']
)

In [52]:
train_df['label'].value_counts()

label
2    1776
1     755
0     411
Name: count, dtype: int64

In [53]:

model_name = "BM-K/KoSimCSE-roberta-multitask"


sbert = SentenceTransformer(model_name)
sbert.max_seq_length = 128

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [54]:
# 3) Dataset 정의 ---------------------------------------------------
# class SBERTDataset(Dataset):
#     def __init__(self, texts, labels ):
#         self.texts = texts
#         self.labels = labels
#         print(len(self.texts), len(self.labels))

#     def __len__(self): 
#         return len(self.labels)
#     def __getitem__(self, idx):
#         with torch.inference_mode():
#             embs = sbert.encode(
#                 self.texts[idx], convert_to_tensor=True, normalize_embeddings=normalize
#             )
#         labels = torch.tensor(self.labels[idx], dtype=torch.long)
#         return embs, labels
# Dataset 정의 
class SBERTDataset(Dataset):
    # 생성자 함수 -> document, labels 받아와서 document 임베딩, labels는 tensor화
    def __init__(self, document, labels):
        # no_grad() -> 자동 미분 일시 정지 
        # inference_mode() -> 추론 모드 
        with torch.inference_mode():
            # 로드한 모델을 이용해서 encode 작업 
            # convet_to_tensor -> 결과값을 tensor로 받을것인가? (False : list)
            # normalize_embeddings -> L2 정규화 할것인가?
            self.emb = sbert.encode(
                document, convert_to_tensor = True, normalize_embeddings=True
            )
        # labels를 tensor화 
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        # labels의 길이를 되돌려준다.
        return len(self.labels)
    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]

In [55]:
train_ds = SBERTDataset(train_df["RawText"].tolist(), train_df["label"].tolist())
test_ds  = SBERTDataset(test_df["RawText"].tolist(),  test_df["label"].tolist())

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=256)

In [56]:
train_ds

In [65]:

# 4) MLP 분류기 -----------------------------------------------------
class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden=256, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x): return self.net(x)

in_dim = sbert.get_sentence_embedding_dimension()
clf = MLPHead(in_dim, num_classes=3)

crit = nn.CrossEntropyLoss(weight=torch.tensor([5.0, 3.0, 1.0]))
opt = torch.optim.AdamW(clf.parameters(), lr=2e-4)


In [66]:

# 5) 학습 -----------------------------------------------------------
clf.train()
for epoch in range(5):
    total = 0.0
    for xb, yb in train_dl:
        xb, yb = xb, yb
        opt.zero_grad()
        logits = clf(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    print(f"epoch {epoch+1}, loss={total/len(train_ds):.4f}")


epoch 1, loss=1.0859
epoch 2, loss=1.0472
epoch 3, loss=0.9939
epoch 4, loss=0.9327
epoch 5, loss=0.8716


In [67]:
# 테스트 데이터를 이용하여 정확도, f1_score 확인
clf.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)  # 예측 데이터 -> [[ 0.xxx, 0.xxxx ], [], [] , ...]
        pred = logits.argmax(dim=1).tolist()    # 예측 데이터 -> [0, 1, 1, 0, ...]
        # y_true에 y를 리스트의 형태로 변환하고 데이터를 확장시킨다. 
        y_true.extend(y.tolist())
        y_pred += pred

In [68]:
print('accuracy_score', accuracy_score(y_true, y_pred))
print('f1_score', f1_score(y_true, y_pred, average='macro'))

accuracy_score 0.6304347826086957
f1_score 0.6395435839880285


In [69]:
samples = na_df['RawText'].sample(10).tolist()
samples

['얼마전에 구매한 티비는 저가의 중소기업 제품의 티비였어요. 티비야 그냥 화면만 잘 나오면 되지 하는 마음에 구매했는데 웬걸 1년이 조금 넘으니 액정에 긴 줄이 나오면서 고장이 나버렸네요. 역시 전자제품은 대기업 제품을 구매해야 하나봐요. 그래서 이번에는 대기업 제품으로 구매했습니다. 믿고 쓰는 OOO전자 제품으로 구매했죠. 역시 OOO 제품은 다르네요. 더 이상 얇을 수 없을 것 같이 얇은 베젤과 깔끔한 디자인 너무 마음에 듭니다. 거실에 놓아두니 티비 하나로 인테리어 같은 효과가 나네요. 잔고장 없이 오래오래 사용할 거라고 믿으며 사용하고 있어요',
 '지난 달에 TV를 구매했어요 몇일동안 머리싸매고 고민하고 구매했는데 2달정도직접 써보니 너무 좋네요 FHD보다 선명한 화질이라 TV시청할 때마다 만족도가 올라가는 것 같습니당. 화질을 중요시하신다면 이제품을 구입하세요 사운드바도 같이 받았는데 정말 영화관에 온 것 같은 느낌이더라구요 생생한 사운드를 즐기실 수 있을거에요 단점으로 아쉬운 건 가격이 조금 더 저렴했으면 좋았을 것 같아요. 제가 구입하고 얼마있지 않아서 프로모션 행사 들어가서 너무 아쉬웠던 기억이 있네요 구입하시기전에 받을 수 있는 혜택이 있으시다면 한번 알아보세요',
 '내돈 내산 후기랍니다 물 용량이 작아도 너무 작네요 손님들이 여러 명 오면 한번에 해결 불가 여러 번 내려야 해서 너무 번거롭네요그리고 원두 가는 소리도 15초면 된다고 하는데 30초이상은 걸리는 것 같네요소음도 너무 커서 출근시간에 내리기에는 이웃들한테 민폐각입니다드립용은 종이 필터가 있어서 깔끔하게 해결되는데 이 제품은 원두 갈고 뒷처리가 개수대에 물을 씻어서 버려야 되서 깔끔하지 못해서 불편하네요원두 갈고 한번에 내리는 간편함에 반해서 샀는데... 거실 인테리어용으로 하나의 예쁜 장식품이 생겨서 고맙네요역시 물건은 보고 사야 된다는 것을 다시 한번 깨닫게 되네요',
 '뜯자마자 박스가 핑크색이라 고급지고 너무 예뻐서 깜짝 놀랐어요 그전에 쓰던건 드라이기인데 10년 정도 

In [71]:

id2label = {
    0 : '부정', 
    1 : '중립', 
    2 : '긍정'
}
# 예측값을 되돌려주는 함수
# model(독립변수)
# 독립변수의 튜닝 -> 
@torch.no_grad()
def predict_review(
    texts, 
    batch_size = 128
):
    # texts : 예측하려고 하는 리뷰의 원문 데이터들
    # batch_size : 묶음의 크기 

    # texts의 정규화 -> texts(list형태) -> map(), for문을 이용하여 정규화 
                #   -> texts(str) -> 1. 문자열을 정규화 함수에 입력, 
                # 2. 문자열이면 리스트의 형태로 변환
    if isinstance(texts, str):
        texts = [texts]
    
    # 정규화 함수에 리뷰 데이터를 넣어준다. 
    texts_norm = [normalize(t) for t in texts]

    # 2개의 모델을 평가모드 전환 clf, sbert3
    sbert.eval()
    clf.eval()

    # 결과 값
    result = []

    # 배치 데이터로 구성 -> encode -> 분류 모델에 데이터 입력 -> 출력 값을 설정 -> result에 대입
    for idx in range(0, len(texts_norm), batch_size):
        batch_texts = texts_norm[ idx : idx + batch_size ]
        # texts = ['a', b', 'c'] 
        # batch_size = 2
        # 첫번째 반복 구간에서는 
        # batch_texts -> ['a', 'b']
        # embs -> 벡터화 -> 열의 개수는 sbert3의 아웃풋의 차원의 수(768) 
        #               -> 행의 개수는 len(batch_texts)
        # probs -> [ [ 0.3, 0.7 ] , [ 0.51, 0.49 ]]
        # preds -> [ 1 , 0 ]
        # sbert3의 encode함수를 이용하여 임베딩 벡터 생성 
        embs = sbert.encode(
            batch_texts, 
            convert_to_tensor=True, 
            normalize_embeddings=True
        )
        # embs를 clf 모델을 이용하여 예측 확률 데이터를 생성 
        logits = clf(embs)
        probs = logits.softmax(dim = -1)
        preds = probs.argmax(dim=-1).tolist()

        for idx2, pred in enumerate(preds):
            # 첫번째 반복문의 1번 루프에서 preds -> [1, 0]
            # 두번째 반복문의 첫번째 루프 
            # idx2 -> 0
            # pred -> 1
            # prob -> prods[0, 1] -> 0.7
            # review -> texts[ 0 + 0 ] -> texts[0]
            # idx2 : 인덱스 
            # pred : 예측 값(예측 확률의 인덱스 - 확률이 높은 곳의 인덱스(0,1))
            # 높은 예측율
            prob = float( probs[idx2, pred] )
            # 리뷰의 원문 
            # 첫번째 반복문의 반복 횟수? -> len(texts) / batch_size + 1
            # 두번째 반복문의 반복 횟수? -> 
            # idx -> 배치의 시작지점
            # idx2 -> 시작점부터 얼마만큼 이동했는가?
            review = texts[idx + idx2]
            # 긍정/부정 라벨링
            label = id2label[pred]
            # prob, review, label들을 result에 추가 
            result.append(
                {
                    'text' : review, 
                    'prob' : prob, 
                    'label' : label
                }
            )
    return result



In [72]:
out_data = predict_review(samples)

In [73]:
out_data

[{'text': '얼마전에 구매한 티비는 저가의 중소기업 제품의 티비였어요. 티비야 그냥 화면만 잘 나오면 되지 하는 마음에 구매했는데 웬걸 1년이 조금 넘으니 액정에 긴 줄이 나오면서 고장이 나버렸네요. 역시 전자제품은 대기업 제품을 구매해야 하나봐요. 그래서 이번에는 대기업 제품으로 구매했습니다. 믿고 쓰는 OOO전자 제품으로 구매했죠. 역시 OOO 제품은 다르네요. 더 이상 얇을 수 없을 것 같이 얇은 베젤과 깔끔한 디자인 너무 마음에 듭니다. 거실에 놓아두니 티비 하나로 인테리어 같은 효과가 나네요. 잔고장 없이 오래오래 사용할 거라고 믿으며 사용하고 있어요',
  'prob': 0.3698302209377289,
  'label': '중립'},
 {'text': '지난 달에 TV를 구매했어요 몇일동안 머리싸매고 고민하고 구매했는데 2달정도직접 써보니 너무 좋네요 FHD보다 선명한 화질이라 TV시청할 때마다 만족도가 올라가는 것 같습니당. 화질을 중요시하신다면 이제품을 구입하세요 사운드바도 같이 받았는데 정말 영화관에 온 것 같은 느낌이더라구요 생생한 사운드를 즐기실 수 있을거에요 단점으로 아쉬운 건 가격이 조금 더 저렴했으면 좋았을 것 같아요. 제가 구입하고 얼마있지 않아서 프로모션 행사 들어가서 너무 아쉬웠던 기억이 있네요 구입하시기전에 받을 수 있는 혜택이 있으시다면 한번 알아보세요',
  'prob': 0.4752052426338196,
  'label': '긍정'},
 {'text': '내돈 내산 후기랍니다 물 용량이 작아도 너무 작네요 손님들이 여러 명 오면 한번에 해결 불가 여러 번 내려야 해서 너무 번거롭네요그리고 원두 가는 소리도 15초면 된다고 하는데 30초이상은 걸리는 것 같네요소음도 너무 커서 출근시간에 내리기에는 이웃들한테 민폐각입니다드립용은 종이 필터가 있어서 깔끔하게 해결되는데 이 제품은 원두 갈고 뒷처리가 개수대에 물을 씻어서 버려야 되서 깔끔하지 못해서 불편하네요원두 갈고 한번에 내리는 간편함에 반해서 샀는데... 거실 